# Avalanche Canada Wildfire Visualisations

This notebook takes places after overlaying the fire polygons from National Burn Area Composite (NBAC) with Avalanche Canada subregions. 

Prior to discovering severe burn patches within NBAC fire polygons, this notebook explores all fires present within selected NBAC data years with Avalanche Canada subregions.

## Imports & Directories

In [ ]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd
import folium

#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from shapely.geometry import Point

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent
data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'
processed_dir = data_dir / 'processed' / 'analysis/'
app_data = data_dir / 'processed' / 'app/'


if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
print(f'Loading Stage A burns shapefiles...')
stage_A_shp = processed_dir / 'stage_A/AvCan_Stage_A/AvCan_Stage_A_all_regions_SHP.shp'
stage_A_burns = gpd.read_file(stage_A_shp)
print(f' Stage A Burns loaded. {stage_A_burns.crs}\n')

print(f'Loading all AvCan fires shapefile...')
AvCan_fires_file = processed_dir / 'avalanche_canada_fires/shapefiles/AvCan_fires_1990_2024.shp'
AvCan_fires = gpd.read_file(AvCan_fires_file)
print(f' AvCan fires loaded. {AvCan_fires.crs}\n')


print(f'Loading Avalanche Canada (AvCan) regions shapefile...')
avcan_path = processed_dir / "avalanche_canada_fires/AvCan_cleaned_subregions.geojson"
avcan_regions= gpd.read_file(avcan_path)
print(f" Avalanche Canada Regions loaded. {avcan_regions.crs}\n")

## Fire perimeters

Identify regions of Stage A fires

In [ ]:
subregions = list(stage_A_burns['subregion'].unique())

regions = list(stage_A_burns['region'].unique())

# AvCan subregions of interest
avcan_reg_sel = avcan_regions[avcan_regions["region"].isin(regions)]

min_year = int(stage_A_burns["year"].min())
max_year = int(stage_A_burns["year"].max())




In [ ]:

# 2. Filter to the AvCan region / subregion you care about --


# 3. Make sure CRS match  -----------------------------------
if avcan_reg_sel.crs != stage_A_burns.crs:
    severe_burn = stage_A_burns.to_crs(avcan_reg_sel.crs)

# 4. Plot overlay  ------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))

# AvCan subregion outline
avcan_reg_sel.boundary.plot(ax=ax, color="black", linewidth=1, label="AvCan Regions")

# Severe burn patch
severe_burn.plot(
    ax=ax,
    color="red",
    alpha=0.6,
    edgecolor="k",
    linewidth=0.8,
    label="Severe burn patch (≥10 ha cluster)",
    
)

ax.set_title("Stage A patches inside AvCan subregions")
ax.legend()
plt.show()


In [ ]:
AvCan_fires = AvCan_fires.rename(columns={
    "year":"Year",
    "tot_adj_ha":"Total Adjusted Area (ha)",
    "region":"Region",
    "subregion":"Subregion",
    'gid':'Unique Fire ID (gid)'
})

AvCan_fires['Total Adjusted Area (ha)'] = AvCan_fires['Total Adjusted Area (ha)'].round(2)

stage_A_burns = stage_A_burns.rename(columns={
    'slp_mn_pct' : 'Slope Mean Percentage',
    'natpark': 'National Park',
    'gid': 'Unique Fire ID (gid)',
    'aspect_car' : 'Majority Cardinal Direction',
    'patch_id': 'Patch ID',
    'patch_area': 'Patch Area (ha)' , 
    'elev_min_m' : 'Min Elevation (m)',
    'year' : 'Year',
    'subregion': 'Subregion',
    'elev_mean_' : 'Mean Elevation (m)',
    'aspect_mea': 'figure this out',
    'region':'Region',
    'elev_max_m': 'Max Elevation (m)',
    'slp_mn_deg' : 'Mean Slope Degree'

})

stage_A_burns['Mean Slope Degree'] = stage_A_burns['Mean Slope Degree'].round(2)
stage_A_burns['Mean Elevation (m)'] = stage_A_burns['Mean Elevation (m)'].round(2)
stage_A_burns['Patch Area (ha)'] = stage_A_burns['Patch Area (ha)'].round(2)

avcan_reg_sel = avcan_reg_sel.rename(columns={
    "region" : "Region",
    'subregion': 'Subregion'
})

# 1. Make a WGS84 (lat/lon) copy for mapping
AvCan_fires_ll = AvCan_fires.to_crs(epsg=4326)   # <--- key line

print("Severe burn CRS for mapping:", AvCan_fires_ll.crs)

In [ ]:
# Mask: inclusive range
mask_year = AvCan_fires_ll["Year"].between(min_year, max_year, inclusive="both")

# Region filter
mask_region = AvCan_fires_ll["Region"].isin(regions)

AvCan_fires_ll_sel = AvCan_fires_ll[mask_region & mask_year]

In [ ]:
# 1. Make a WGS84 (lat/lon) copy for mapping
stage_A_burns_ll = stage_A_burns.to_crs(epsg=4326)   # <--- key line

print("Severe burn CRS for mapping:", stage_A_burns_ll.crs)

# centre on all severe_burn patches
centroid = stage_A_burns_ll.iloc[0].geometry.centroid
center = [centroid.y, centroid.x]   # [lat, lon]

m = avcan_reg_sel.explore(
    tiles="OpenTopoMap",  # or "OpenStreetMap", "Stamen Terrain", ...
    style_kwds=dict(color="black", weight=2, fill=False),
    name="AvCan Regions",
    location=center,   # <- initial centre
    zoom_start=14,     # <- tweak until it feels right
    width=900,
    height=600,
)

fire_cols = ["Region","Subregion","Unique Fire ID (gid)","Year","Total Adjusted Area (ha)"]

AvCan_fires_ll_sel.explore(
    m=m,
    color="orange",
    name="Fire Polygon",
    tooltip=fire_cols,
    popup=fire_cols
)

stage_A_cols = ["Region","Subregion","Unique Fire ID (gid)", "Patch ID", "Year", "Majority Cardinal Direction","Patch Area (ha)", "Mean Elevation (m)","Mean Slope Degree"]  # whatever you want visible
stage_A_burns_ll.explore(
    m=m,
    color="red",
    name="Stage A burn patches",
    tooltip=stage_A_cols,   # shown on hover
    popup=stage_A_cols      # shown on click
    
)


page_title = "Stage A fires Burn Tree Zone Patches"

m.get_root().html.add_child(
    folium.Element(f"<title>{page_title}</title>")
)

m  # show map


In [ ]:
m.save(REPO_ROOT / f"docs/Stage_A_Zone_Severity.html")

In [ ]:
sel_regions = stage_A_burns_ll['Region'].unique()

print(f'Below is the number of polygons for each layer for the assessed AvCan regions: {sel_regions}\n')

print(f'Number of Fires across years {AvCan_fires_ll_sel['Year'].min()} - {AvCan_fires_ll_sel['Year'].max()}: {AvCan_fires_ll_sel.shape[0]} ')
print(f'Number of Stage A Severity Patches across years {stage_A_burns_ll['Year'].min()} - {stage_A_burns_ll['Year'].max()}: {stage_A_burns_ll.shape[0]}')

In [ ]:
# Export to GeoPackage
# The driver='GPKG' argument is crucial for specifying the format

patches_path = app_data / 'Stage_A_Patches.GPKG'
stage_A_burns_ll.to_file(patches_path, driver='GPKG', layer='Stage A Patches' )

fires_path = app_data / 'Stage_A_Fires.GPKG'
AvCan_fires_ll_sel.to_file(fires_path, driver='GPKG', layer='Stage A Fires')

avcan_regions_path = app_data / "AvCan_Regions.GPKG"
avcan_reg_sel.to_file(avcan_regions_path, driver= 'GPKG', layer='Stage A AvCan Regions')

print(f"Successfully exported data to {app_data}")